In [1]:
import psycopg2
import pandas as pd

# Подключаемся к PostgreSQL в Docker
conn = psycopg2.connect(
    host='localhost',
    port=5433,
    database='practice_db',
    user='postgres',
    password='mysecretpassword'
)

print("Подключение успешно!")

# Создаём таблицу
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS github_repos (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100),
        language VARCHAR(50),
        stars INT,
        created_at TIMESTAMP
    )
""")
conn.commit()
print("Таблица создана!")

Подключение успешно!
Таблица создана!


In [2]:
import requests

# Получаем репозитории с GitHub API
response = requests.get('https://api.github.com/users/slk-oss/repos')
repos = response.json()

# Вставляем в PostgreSQL
for repo in repos:
    cursor.execute("""
        INSERT INTO github_repos (name, language, stars, created_at)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT DO NOTHING
    """, (
        repo['name'],
        repo['language'],
        repo['stargazers_count'],
        repo['created_at']
    ))

conn.commit()
print(f"Загружено {len(repos)} репозиториев в базу")

Загружено 5 репозиториев в базу


In [3]:
# Читаем данные из PostgreSQL в DataFrame
df = pd.read_sql("SELECT * FROM github_repos", conn)
print(df)

   id                          name          language  stars  \
0   1  cpp-diagonal-matrix-iterator               C++      0   
1   2          data-analyst-journey  Jupyter Notebook      0   
2   3                     greatness             Swift      1   
3   4                     HabitHero             Swift      2   
4   5                       slk-oss               NaN      0   

           created_at  
0 2026-03-31 22:10:23  
1 2026-04-07 00:54:24  
2 2026-03-30 20:08:52  
3 2025-11-01 19:09:01  
4 2026-03-31 12:18:29  


/var/folders/j3/2yy7s8z57tl78m366kkwbdqr0000gn/T/ipykernel_98933/3760599911.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM github_repos", conn)
